# Baseline Model Revised (Development Subset + Time Split)

對應 `strategy/baseline_model_revised.md` 的實作版 notebook。

重點：
1. 固定 time-based split（70/15/15 by unique steps）
2. 分階段訓練資料（Stage A/B/C）
3. 模型比較：Logistic / Decision Tree / LightGBM / XGBoost（可選）
4. 指標以 PR-AUC + threshold scan + risk tier 為主

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    average_precision_score, roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

# Optional boosting models
try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

## 0) Config

In [ ]:
DATA_PATH = '../Transactions Data.csv'
TARGET = 'isFraud'
RANDOM_STATE = 42

# Fixed time split ratios
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

# Stage: 'A' | 'B' | 'C'
STAGE = 'A'
STAGE_TRAIN_FRAC = {
    'A': 0.20,   # 快速迭代
    'B': 0.50,   # 穩健驗證
    'C': 1.00    # 最終版（資源允許）
}

# Model toggles
RUN_LOGIT = True
RUN_DTREE = True
RUN_LIGHTGBM = True
RUN_XGBOOST = True

# Sensitive feature ablation
USE_FLAGGED_FEATURE = True

# Threshold scan
THRESHOLDS = (0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50)

## 1) Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values('step').reset_index(drop=True)
print('Shape:', df.shape)
print('Fraud rate:', df[TARGET].mean())
df.head(3)

## 2) Feature Engineering

In [ ]:
work = df.copy()
work['deltaOrig'] = work['oldbalanceOrg'] - work['newbalanceOrig']
work['deltaDest'] = work['newbalanceDest'] - work['oldbalanceDest']
work['isOrigBalanceZero'] = (work['oldbalanceOrg'] == 0).astype(int)
work['isDestBalanceZero'] = (work['oldbalanceDest'] == 0).astype(int)

# baseline 先移除高基數 ID
work = work.drop(columns=['nameOrig', 'nameDest'])

if not USE_FLAGGED_FEATURE and 'isFlaggedFraud' in work.columns:
    work = work.drop(columns=['isFlaggedFraud'])

num_features = [
    c for c in [
        'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
        'oldbalanceDest', 'newbalanceDest', 'deltaOrig', 'deltaDest',
        'isOrigBalanceZero', 'isDestBalanceZero', 'isFlaggedFraud'
    ] if c in work.columns
]
cat_features = ['type']

feature_cols = [c for c in work.columns if c != TARGET]
X = work[feature_cols]
y = work[TARGET]

print('Num features:', num_features)
print('Cat features:', cat_features)
print('Total features used:', len(feature_cols))

## 3) Time-based Split (Fixed Framework)

In [ ]:
steps_sorted = np.sort(work['step'].unique())
train_cut_idx = int(len(steps_sorted) * TRAIN_RATIO)
valid_cut_idx = int(len(steps_sorted) * (TRAIN_RATIO + VALID_RATIO))

train_max_step = steps_sorted[train_cut_idx - 1]
valid_max_step = steps_sorted[valid_cut_idx - 1]

train_mask = work['step'] <= train_max_step
valid_mask = (work['step'] > train_max_step) & (work['step'] <= valid_max_step)
test_mask = work['step'] > valid_max_step

X_train_full, y_train_full = X.loc[train_mask], y.loc[train_mask]
X_valid, y_valid = X.loc[valid_mask], y.loc[valid_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print('Step boundary: train<=', train_max_step, ', valid<=', valid_max_step)
print('Train full:', X_train_full.shape, 'Fraud rate:', y_train_full.mean())
print('Valid     :', X_valid.shape, 'Fraud rate:', y_valid.mean())
print('Test      :', X_test.shape, 'Fraud rate:', y_test.mean())

## 4) Stage-wise Train Subset (A/B/C)

In [ ]:
train_frac = STAGE_TRAIN_FRAC[STAGE]

if train_frac < 1.0:
    # 在 train 區間內做分層抽樣，保持 fraud 比例
    train_df = X_train_full.copy()
    train_df[TARGET] = y_train_full.values

    sampled_train = (
        train_df.groupby(TARGET, group_keys=False)
                .apply(lambda g: g.sample(frac=train_frac, random_state=RANDOM_STATE))
                .sample(frac=1, random_state=RANDOM_STATE)
                .reset_index(drop=True)
    )

    X_train = sampled_train.drop(columns=[TARGET])
    y_train = sampled_train[TARGET]
else:
    X_train, y_train = X_train_full, y_train_full

print(f'STAGE={STAGE}, train_frac={train_frac}')
print('Train used:', X_train.shape, 'Fraud rate:', y_train.mean())

## 5) Preprocessors

In [ ]:
preprocess_lr = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features),
    ],
    remainder='drop'
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features),
    ],
    remainder='drop'
)

## 6) Helpers

In [ ]:
def threshold_table(y_true, y_prob, thresholds=THRESHOLDS):
    rows = []
    y_true = np.asarray(y_true)
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append((t, precision, recall, f1, int((y_pred == 1).sum())))

    return pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'f1', 'predicted_positive'])


def evaluate_probs(y_true, y_prob, model_name='model', threshold=0.5):
    y_true = np.asarray(y_true)
    y_pred = (y_prob >= threshold).astype(int)

    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)

    print(f'[{model_name}] threshold={threshold:.3f}')
    print(f'PR-AUC: {pr_auc:.6f}')
    print(f'ROC-AUC: {roc_auc:.6f}')
    print('Confusion Matrix:\n', confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))

    return {'model': model_name, 'threshold': threshold, 'pr_auc': pr_auc, 'roc_auc': roc_auc}


def plot_pr_curve(y_true, y_prob, title='PR Curve'):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.figure(figsize=(6, 4))
    plt.plot(r, p, label=f'AP={ap:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


def assign_risk_tier(prob, t_mid=0.10, t_high=0.30):
    return pd.cut(prob, bins=[-np.inf, t_mid, t_high, np.inf], labels=['Low', 'Medium', 'High'])

## 7) Train & Evaluate Models

In [ ]:
results = []
valid_probs = {}
test_probs = {}

In [ ]:
if RUN_LOGIT:
    logit = Pipeline([
        ('prep', preprocess_lr),
        ('clf', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            solver='saga',
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    logit.fit(X_train, y_train)
    valid_probs['logit'] = logit.predict_proba(X_valid)[:, 1]
    test_probs['logit'] = logit.predict_proba(X_test)[:, 1]

    plot_pr_curve(y_valid, valid_probs['logit'], 'Logistic - Valid PR Curve')
    m = evaluate_probs(y_valid, valid_probs['logit'], model_name='Logistic (Valid)', threshold=0.5)
    results.append(m)
    display(threshold_table(y_valid, valid_probs['logit']))

In [ ]:
if RUN_DTREE:
    dtree = Pipeline([
        ('prep', preprocess_tree),
        ('clf', DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=200,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ])

    dtree.fit(X_train, y_train)
    valid_probs['dtree'] = dtree.predict_proba(X_valid)[:, 1]
    test_probs['dtree'] = dtree.predict_proba(X_test)[:, 1]

    plot_pr_curve(y_valid, valid_probs['dtree'], 'Decision Tree - Valid PR Curve')
    m = evaluate_probs(y_valid, valid_probs['dtree'], model_name='DecisionTree (Valid)', threshold=0.5)
    results.append(m)
    display(threshold_table(y_valid, valid_probs['dtree']))

In [ ]:
if RUN_LIGHTGBM:
    if not HAS_LIGHTGBM:
        print('LightGBM not installed. Run: uv add lightgbm')
    else:
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        scale_pos_weight = (neg / pos) if pos > 0 else 1.0

        lgbm = Pipeline([
            ('prep', preprocess_tree),
            ('clf', LGBMClassifier(
                objective='binary',
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                min_child_samples=100,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ])

        lgbm.fit(X_train, y_train)
        valid_probs['lgbm'] = lgbm.predict_proba(X_valid)[:, 1]
        test_probs['lgbm'] = lgbm.predict_proba(X_test)[:, 1]

        plot_pr_curve(y_valid, valid_probs['lgbm'], 'LightGBM - Valid PR Curve')
        m = evaluate_probs(y_valid, valid_probs['lgbm'], model_name='LightGBM (Valid)', threshold=0.5)
        results.append(m)
        display(threshold_table(y_valid, valid_probs['lgbm']))

In [ ]:
if RUN_XGBOOST:
    if not HAS_XGBOOST:
        print('XGBoost not installed. Run: uv add xgboost')
    else:
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        scale_pos_weight = (neg / pos) if pos > 0 else 1.0

        xgb = Pipeline([
            ('prep', preprocess_tree),
            ('clf', XGBClassifier(
                objective='binary:logistic',
                eval_metric='aucpr',
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                min_child_weight=5,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                scale_pos_weight=scale_pos_weight
            ))
        ])

        xgb.fit(X_train, y_train)
        valid_probs['xgb'] = xgb.predict_proba(X_valid)[:, 1]
        test_probs['xgb'] = xgb.predict_proba(X_test)[:, 1]

        plot_pr_curve(y_valid, valid_probs['xgb'], 'XGBoost - Valid PR Curve')
        m = evaluate_probs(y_valid, valid_probs['xgb'], model_name='XGBoost (Valid)', threshold=0.5)
        results.append(m)
        display(threshold_table(y_valid, valid_probs['xgb']))

## 8) Validation Comparison

In [ ]:
valid_compare = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
display(valid_compare)

## 9) Select Model + Evaluate on Test

In [ ]:
if len(valid_compare) == 0:
    raise ValueError('No model results found. Please enable at least one model.')

best_model_key_map = {
    'Logistic (Valid)': 'logit',
    'DecisionTree (Valid)': 'dtree',
    'LightGBM (Valid)': 'lgbm',
    'XGBoost (Valid)': 'xgb'
}

best_model_name = valid_compare.iloc[0]['model']
best_key = best_model_key_map[best_model_name]

# 可自行改成 business-driven threshold
SELECT_THRESHOLD = 0.10

print('Selected model:', best_model_name, '| key:', best_key)
test_metric = evaluate_probs(y_test, test_probs[best_key], model_name=best_model_name.replace('(Valid)','(Test)'), threshold=SELECT_THRESHOLD)
display(threshold_table(y_test, test_probs[best_key]))

## 10) Risk Tier Report

In [ ]:
T_MID = 0.10
T_HIGH = 0.30

risk_df = pd.DataFrame({
    'y_true': y_test.values,
    'prob': test_probs[best_key]
})
risk_df['risk_tier'] = assign_risk_tier(risk_df['prob'], t_mid=T_MID, t_high=T_HIGH)

tier_summary = risk_df.groupby('risk_tier', observed=True).agg(
    volume=('y_true', 'size'),
    fraud_count=('y_true', 'sum'),
    fraud_rate=('y_true', 'mean')
).reset_index().sort_values('risk_tier')
tier_summary['volume_pct'] = tier_summary['volume'] / tier_summary['volume'].sum()
tier_summary['capture_pct'] = tier_summary['fraud_count'] / tier_summary['fraud_count'].sum()
display(tier_summary)

## 11) Notes for Interview Narrative

- 先用 Stage A 快速迭代特徵與指標框架
- 維持固定 time split，確保方法正確
- 再往 Stage B/C 放大訓練量，觀察 PR-AUC 與 Recall@Precision 是否穩定
- 最終以 risk tier 可操作性（審核量 vs fraud capture）做決策